# Chapter 7 - Demo
**Working with Keras: A deep dive**  
**Làm việc chuyên sâu với Keras**

Notebook này minh họa các ý quan trọng trong **Chapter 7**:

- **7.1 A spectrum of workflows:** Keras có nhiều mức workflow, từ API rất tiện lợi đến tự viết training loop.
- **7.2 Different ways to build Keras models:** `Sequential`, Functional API, model subclassing, chia sẻ layer và inspect model.
- **7.3 Using built-in training and evaluation loops:** `compile()`, `fit()`, `evaluate()`, `predict()`, custom metric và callbacks.
- **7.4 Writing your own training and evaluation loops:** dùng `GradientTape`, metric ở mức thấp, evaluation loop, `tf.function` và custom `train_step()`.

Dữ liệu trong notebook là **dữ liệu giả lập nhỏ**, giúp sinh viên tập trung vào API Keras thay vì xử lý dữ liệu phức tạp.

## Chuẩn bị môi trường

Cell dưới đây import TensorFlow/Keras, NumPy và đặt random seed để kết quả dễ tái lập. Nếu máy chưa cài TensorFlow, cần cài trước khi chạy notebook.

In [14]:
import os
import shutil

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Cố định seed để các lần chạy cho kết quả gần giống nhau.
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Tạo dữ liệu giả lập

Ta tạo hai nhóm dữ liệu:

1. **Dữ liệu vector đơn giản** để demo model phân loại nhiều lớp.
2. **Dữ liệu ticket hỗ trợ khách hàng** để demo Functional API với nhiều input và nhiều output.

Trong thực tế, dữ liệu ticket có thể gồm tiêu đề, nội dung, tag, người gửi, thời gian gửi, v.v. Ở đây ta dùng vector ngẫu nhiên để giữ ví dụ ngắn gọn.

In [16]:
# Dữ liệu phân loại vector: mỗi mẫu có 20 feature và thuộc 1 trong 3 lớp.
num_samples = 1024
num_features = 20
num_classes = 3

# Sinh ma trận đặc trưng giả lập theo phân phối chuẩn.
x = np.random.normal(size=(num_samples, num_features)).astype("float32")

# Tạo một bộ trọng số ẩn để sinh nhãn có quy luật, không hoàn toàn ngẫu nhiên.
true_w = np.random.normal(size=(num_features, num_classes)).astype("float32")

# Logit là điểm thô cho từng lớp; phần nhiễu giúp dữ liệu giống thực tế hơn.
# Phép nhân ma trận x @ true_w tạo điểm cho từng lớp của từng mẫu, rồi cộng nhiễu Gaussian nhỏ.
# x -> logits -> probabilities -> class label
# x -> logits: để chuyển từ không gian feature sang không gian class
logits = x @ true_w + 0.3 * np.random.normal(size=(num_samples, num_classes))
# Chọn lớp có logit lớn nhất làm nhãn mục tiêu; int32 phù hợp với sparse_categorical_crossentropy.
y = np.argmax(logits, axis=1).astype("int32")

# Chia dữ liệu thành tập train và validation.
x_train, x_val = x[:800], x[800:]
y_train, y_val = y[:800], y[800:]

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_val:", x_val.shape)
print("y_val:", y_val.shape)

# Dữ liệu ticket giả lập cho bài toán nhiều input, nhiều output.
ticket_samples = 1000

# title_data, body_data và tags_data mô phỏng các nhóm đặc trưng khác nhau của ticket.
title_data = np.random.randint(0, 2, size=(ticket_samples, 100)).astype("float32")
body_data = np.random.randint(0, 2, size=(ticket_samples, 1000)).astype("float32")
tags_data = np.random.randint(0, 2, size=(ticket_samples, 12)).astype("float32")

# Output 1: mức ưu tiên, dạng số thực trong khoảng [0, 1].
priority_targets = np.random.random(size=(ticket_samples, 1)).astype("float32")

# Output 2: phòng ban xử lý, gồm 4 lớp rời rạc.
department_targets = np.random.randint(0, 4, size=(ticket_samples,)).astype("int32")

print("title_data:", title_data.shape)
print("body_data:", body_data.shape)
print("tags_data:", tags_data.shape)
print("priority_targets:", priority_targets.shape)
print("department_targets:", department_targets.shape)

x_train: (800, 20)
y_train: (800,)
x_val: (224, 20)
y_val: (224,)
title_data: (1000, 100)
body_data: (1000, 1000)
tags_data: (1000, 12)
priority_targets: (1000, 1)
department_targets: (1000,)


# 7.1 A spectrum of workflows - Phổ các quy trình làm việc

Keras cho phép làm việc ở nhiều mức:

- **Mức cao:** dùng `Sequential`, `compile()`, `fit()` cho model đơn giản.
- **Mức trung bình:** dùng Functional API để xây model nhiều nhánh, nhiều input/output.
- **Mức linh hoạt:** subclass `keras.Model` khi cần logic Python tùy biến.
- **Mức thấp:** tự viết training loop bằng `tf.GradientTape` khi thuật toán train không theo mẫu chuẩn.

Nguyên tắc thực hành: **dùng mức trừu tượng cao nhất vẫn giải quyết được bài toán**.

In [17]:
# Bảng tóm tắt giúp so sánh các mức workflow trong Keras.
workflow_table = [
    ("Sequential + fit", "Nhanh, đơn giản", "Model tuyến tính, một input, một output"),
    ("Functional API", "Linh hoạt nhưng vẫn inspect được", "Nhiều input/output, skip connection, shared layer"),
    ("Model subclassing", "Tùy biến kiến trúc bằng Python", "Logic động, điều kiện, vòng lặp"),
    ("Custom training loop", "Kiểm soát sâu quá trình train", "Thuật toán train đặc biệt, nhiều optimizer"),
]

for api, strength, use_case in workflow_table:
    print(f"{api:22s} | {strength:35s} | {use_case}")

Sequential + fit       | Nhanh, đơn giản                     | Model tuyến tính, một input, một output
Functional API         | Linh hoạt nhưng vẫn inspect được    | Nhiều input/output, skip connection, shared layer
Model subclassing      | Tùy biến kiến trúc bằng Python      | Logic động, điều kiện, vòng lặp
Custom training loop   | Kiểm soát sâu quá trình train       | Thuật toán train đặc biệt, nhiều optimizer


# 7.2 Different ways to build Keras models
# 7.2 Các cách xây dựng mô hình Keras

Phần này demo ba cách xây model:

1. **Sequential model** - đơn giản nhất.
2. **Functional API** - phù hợp với model dạng đồ thị.
3. **Model subclassing** - linh hoạt nhất về kiến trúc.

Ngoài ra, ta sẽ demo thêm **layer sharing** và **inspect intermediate features**.

## 7.2.1 Sequential model - Mô hình tuần tự

`Sequential` phù hợp khi model là một chuỗi layer tuyến tính: input đi qua layer 1, rồi layer 2, rồi output. Đây là lựa chọn tốt để tạo baseline nhanh.

In [18]:
def make_sequential_classifier():
    """Tạo mô hình phân loại nhiều lớp bằng Sequential API.

    Tham số:
        Không có. Hàm dùng các biến toàn cục num_features và num_classes
        đã được khai báo ở phần chuẩn bị dữ liệu.

    Kiểu trả về:
        keras.Sequential: mô hình nhận vector đặc trưng và trả về xác suất
        của từng lớp.
    """
    return keras.Sequential(
        [
            # Khai báo shape đầu vào để model biết số feature của mỗi mẫu.
            layers.Input(shape=(num_features,), name="features"),

            # Hai lớp Dense học biểu diễn phi tuyến từ dữ liệu vector.
            layers.Dense(64, activation="relu", name="dense_1"),
            layers.Dense(64, activation="relu", name="dense_2"),

            # Softmax biến output thành phân phối xác suất trên các lớp.
            layers.Dense(num_classes, activation="softmax", name="predictions"),
        ],
        name="sequential_classifier",
    )


seq_model = make_sequential_classifier()
seq_model.summary()

Model: "sequential_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,699 (22.26 KB)

 Trainable params: 5,699 (22.26 KB)

 Non-trainable params: 0 (0.00 B)

## 7.2.2 Functional API - Tư duy model như đồ thị tensor

Functional API bắt đầu bằng `keras.Input`, sau đó gọi layer như hàm trên tensor. Cách này phù hợp với:

- Model có **nhiều input**.
- Model có **nhiều output**.
- Model có **nhánh rẽ**, **ghép nhánh**, hoặc **shared layer**.

Ví dụ dưới đây mô phỏng hệ thống xử lý ticket hỗ trợ khách hàng.

In [19]:
# Ba input đại diện cho ba nguồn thông tin của một ticket.
# Mỗi title là một vector 100 chiều
title_input = keras.Input(shape=(100,), name="title")
body_input = keras.Input(shape=(1000,), name="body")
tags_input = keras.Input(shape=(12,), name="tags")

# Mỗi input có thể đi qua nhánh xử lý riêng để học biểu diễn phù hợp.
# title -> xử lý -> title_features  \
# body  -> xử lý -> body_features    -> concatenate -> Dense -> output
# tags  ----------------------------/
title_features = layers.Dense(64, activation="relu", name="title_features")(title_input)
body_features = layers.Dense(64, activation="relu", name="body_features")(body_input)

# Ghép đặc trưng từ các nhánh để tạo biểu diễn chung cho toàn bộ ticket.
x_ticket = layers.Concatenate(name="concatenate_features")(
    [title_features, body_features, tags_input]
)
x_ticket = layers.Dense(64, activation="relu", name="shared_features")(x_ticket)

# Output priority là bài toán hồi quy nhị phân hóa trong khoảng [0, 1].
priority_output = layers.Dense(1, activation="sigmoid", name="priority")(x_ticket)

# Output department là bài toán phân loại 4 lớp.
department_output = layers.Dense(4, activation="softmax", name="department")(x_ticket)

# Functional API cho phép gom nhiều input và nhiều output vào một model.
ticket_model = keras.Model(
    inputs=[title_input, body_input, tags_input],
    outputs=[priority_output, department_output],
    name="ticket_routing_model",
)

ticket_model.summary()

Model: "ticket_routing_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title (InputLayer)  │ (None, 100)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ body (InputLayer)   │ (None, 1000)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_features      │ (None, 64)        │      6,464 │ title[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ body_features       │ (None, 64)        │     64,064 │ body[0][0]        │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tags (InputLayer)   │ (None, 12)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_featur… │ (None, 140)       │          0 │ title_features[0… │
│ (Concatenate)       │                   │            │ body_features[0]… │
│                     │                   │            │ tags[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_features     │ (None, 64)        │      9,024 │ concatenate_feat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ priority (Dense)    │ (None, 1)         │         65 │ shared_features[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department (Dense)  │ (None, 4)         │        260 │ shared_features[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 79,877 (312.02 KB)

 Trainable params: 79,877 (312.02 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Compile model nhiều output: mỗi output có loss và metric riêng.
ticket_model.compile(
    optimizer="rmsprop",
    loss={
        # priority là giá trị liên tục nên dùng MSE.
        "priority": "mse",
        # department là nhãn số nguyên nên dùng sparse categorical crossentropy.
        "department": "sparse_categorical_crossentropy",
    },
    metrics={
        "priority": ["mae"],
        "department": ["accuracy"],
    },
)

# Truyền dữ liệu bằng dictionary để map đúng tên input và output.
history_ticket = ticket_model.fit(
    {"title": title_data, "body": body_data, "tags": tags_data},
    {"priority": priority_targets, "department": department_targets},
    epochs=2,
    batch_size=32,
    verbose=0,
)

print("Các khóa trong history:")
for key in history_ticket.history.keys():
    print("-", key)

Các khóa trong history:
- department_accuracy
- department_loss
- loss
- priority_loss
- priority_mae


## Inspect model - Kiểm tra kết nối và lấy đặc trưng trung gian

Với Functional API, Keras biết rõ đồ thị kết nối. Ta có thể:

- Xem danh sách layer.
- Lấy output của một layer trung gian.
- Tạo model mới dùng lại input cũ nhưng output là activation trung gian.

Điều này rất hữu ích khi debug hoặc trực quan hóa feature.

In [21]:
print("Danh sách layer trong ticket_model:")
for layer in ticket_model.layers:
    # output_shape giúp kiểm tra kích thước tensor sau từng layer.
    output_shape = layer.output_shape if hasattr(layer, "output_shape") else "dynamic"
    print(f"{layer.name:25s} | output shape: {output_shape}")

# Tạo model trung gian lấy output của layer shared_features.
# Model này dùng lại input cũ nhưng trả về activation ở giữa mạng.
feature_extractor = keras.Model(
    inputs=ticket_model.inputs,
    outputs=ticket_model.get_layer("shared_features").output,
    name="ticket_feature_extractor",
)

# training=False bảo đảm các layer có hành vi train/inference sẽ chạy ở chế độ inference.
sample_features = feature_extractor(
    [title_data[:3], body_data[:3], tags_data[:3]],
    training=False,
)

print("Shape của feature trung gian:", sample_features.shape)

Danh sách layer trong ticket_model:
title                     | output shape: dynamic
body                      | output shape: dynamic
title_features            | output shape: dynamic
body_features             | output shape: dynamic
tags                      | output shape: dynamic
concatenate_features      | output shape: dynamic
shared_features           | output shape: dynamic
priority                  | output shape: dynamic
department                | output shape: dynamic
Shape của feature trung gian: (3, 64)


## Layer sharing - Chia sẻ layer

Trong Functional API, nếu cùng một layer object được gọi nhiều lần, các lần gọi đó **dùng chung weight**. Đây là ý tưởng quan trọng trong shared encoder hoặc siamese network.

In [22]:
# Hai input có cùng shape để đi qua cùng một encoder.
input_a = keras.Input(shape=(10,), name="item_a")
input_b = keras.Input(shape=(10,), name="item_b")

# Đây là một layer object duy nhất; nếu gọi nhiều lần thì các lần gọi dùng chung weight.
shared_encoder = layers.Dense(16, activation="relu", name="shared_encoder")

# Gọi cùng shared_encoder trên hai input khác nhau để tạo mô hình kiểu siamese.
encoded_a = shared_encoder(input_a)
encoded_b = shared_encoder(input_b)

# Tính độ tương đồng giữa hai vectors [encoded_a, encoded_b] bằng chỉ số cosine_similarity
# Dot với normalize=True cho ra cosine similarity giữa hai vector đã mã hóa.
# axes=1 nghĩa là tính dot product theo chiều số 1, tức chiều đặc trưng
similarity = layers.Dot(axes=1, normalize=True, name="cosine_similarity")(
    [encoded_a, encoded_b]
)

siamese_model = keras.Model([input_a, input_b], similarity, name="siamese_demo")
siamese_model.summary()

print("Số weight object trong shared_encoder:", len(shared_encoder.weights))

Model: "siamese_demo"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ item_a (InputLayer) │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_b (InputLayer) │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_encoder      │ (None, 16)        │        176 │ item_a[0][0],     │
│ (Dense)             │                   │            │ item_b[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity   │ (None, 1)         │          0 │ shared_encoder[0… │
│ (Dot)               │                   │            │ shared_encoder[1… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 176 (704.00 B)

 Trainable params: 176 (704.00 B)

 Non-trainable params: 0 (0.00 B)

Số weight object trong shared_encoder: 2


## 7.2.3 Model subclassing - Kế thừa `keras.Model`

Subclassing phù hợp khi kiến trúc cần logic Python tùy biến. Ta khai báo layer trong `__init__()` và viết forward pass trong `call()`.

Nhược điểm: Keras khó inspect toàn bộ graph tự động như Functional API.

In [23]:
class MLPClassifier(keras.Model):
    """Mô hình MLP phân loại nhiều lớp viết bằng model subclassing.

    Tham số:
        num_classes (int): số lớp cần dự đoán ở output.

    Kiểu trả về khi gọi model:
        tf.Tensor: tensor xác suất có shape (batch_size, num_classes).
    """

    def __init__(self, num_classes=3):
        """Khai báo các layer sẽ được dùng trong forward pass."""
        super().__init__(name="subclassed_mlp")
        self.dense_1 = layers.Dense(64, activation="relu")
        self.dropout = layers.Dropout(0.3)
        self.dense_2 = layers.Dense(64, activation="relu")
        self.classifier = layers.Dense(num_classes, activation="softmax")

    def call(self, inputs, training=False):
        """Chạy forward pass.

        Tham số:
            inputs (tf.Tensor): batch dữ liệu đầu vào, shape (batch_size, num_features).
            training (bool): True khi huấn luyện, False khi suy luận.

        Kiểu trả về:
            tf.Tensor: xác suất dự đoán cho từng lớp.
        """
        x = self.dense_1(inputs)
        x = self.dropout(x, training=training)
        x = self.dense_2(x)
        return self.classifier(x)


subclassed_model = MLPClassifier(num_classes=num_classes)

# Gọi model một lần để Keras tạo weights trước khi in summary.
_ = subclassed_model(tf.zeros((1, num_features)))
subclassed_model.summary()

Model: "subclassed_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (1, 64)                │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (1, 64)                │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (1, 3)                 │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,699 (22.26 KB)

 Trainable params: 5,699 (22.26 KB)

 Non-trainable params: 0 (0.00 B)

# 7.3 Dùng vòng lặp huấn luyện và đánh giá có sẵn

Workflow chuẩn trong Keras:

1. `compile()` - cấu hình optimizer, loss, metrics.
2. `fit()` - huấn luyện.
3. `evaluate()` - đánh giá.
4. `predict()` - suy luận.

Đây nên là lựa chọn mặc định cho phần lớn dự án, vì Keras đã xử lý batching, validation, logs, callbacks và history.

In [24]:
fit_model = make_sequential_classifier()

# compile cấu hình thuật toán tối ưu, hàm mất mát và metric theo dõi.
fit_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

# fit thực hiện toàn bộ training loop mức cao: chia batch, forward, backward và cập nhật metric.
history = fit_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=0,
)

print("History keys:", history.history.keys())
print("Final train accuracy:", history.history["accuracy"][-1])
print("Final val accuracy:", history.history["val_accuracy"][-1])

# evaluate trả về loss và metric trên tập validation.
val_loss, val_acc = fit_model.evaluate(x_val, y_val, verbose=0)
print("Evaluate - val_loss:", round(val_loss, 4), "val_acc:", round(val_acc, 4))

# predict trả về xác suất dự đoán; argmax chuyển xác suất thành nhãn lớp.
predictions = fit_model.predict(x_val[:5], verbose=0)
print("Shape dự đoán cho 5 mẫu:", predictions.shape)
print("Lớp dự đoán:", np.argmax(predictions, axis=1))

History keys: dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])
Final train accuracy: 0.8987500071525574
Final val accuracy: 0.8348214030265808
Evaluate - val_loss: 0.3951 val_acc: 0.8348
Shape dự đoán cho 5 mẫu: (5, 3)
Lớp dự đoán: [0 0 1 1 1]


## Custom metric - Tự viết metric

Metric là đại lượng theo dõi chất lượng model. Một metric có state cần các thành phần:

- `add_weight()` để tạo biến trạng thái.
- `update_state()` để cập nhật state trên từng batch.
- `result()` để trả về giá trị metric.
- `reset_state()` để làm sạch state trước epoch/evaluation mới.

In [25]:
class RootMeanSquaredError(keras.metrics.Metric):
    """Metric RMSE tự viết theo chuẩn keras.metrics.Metric.

    Tham số:
        name (str): tên metric hiển thị trong log huấn luyện.
        **kwargs: các tham số bổ sung do Keras truyền vào Metric.

    Kiểu trả về của result:
        tf.Tensor: giá trị RMSE hiện tại.
    """

    def __init__(self, name="rmse", **kwargs):
        super().__init__(name=name, **kwargs)
        # squared_sum lưu tổng bình phương sai số qua các batch đã thấy.
        self.squared_sum = self.add_weight(name="squared_sum", initializer="zeros")
        # total lưu tổng số phần tử đã dùng để tính metric.
        self.total = self.add_weight(name="total", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        """Cập nhật trạng thái metric bằng một batch dữ liệu."""
        y_true = tf.cast(y_true, y_pred.dtype)
        error = tf.square(y_true - y_pred)
        self.squared_sum.assign_add(tf.reduce_sum(error))
        self.total.assign_add(tf.cast(tf.size(error), tf.float32))

    def result(self):
        """Tính RMSE từ trạng thái tích lũy hiện tại."""
        return tf.sqrt(self.squared_sum / self.total)

    def reset_state(self):
        """Đưa metric về trạng thái rỗng, thường gọi sau mỗi epoch."""
        self.squared_sum.assign(0.0)
        self.total.assign(0.0)


rmse = RootMeanSquaredError()
rmse.update_state(tf.constant([1.0, 2.0, 3.0]), tf.constant([1.2, 1.8, 2.5]))
print("RMSE:", float(rmse.result()))
rmse.reset_state()
print("RMSE sau reset:", float(rmse.result().numpy()) if rmse.total.numpy() != 0 else "chưa có dữ liệu")

RMSE: 0.33166250586509705
RMSE sau reset: chưa có dữ liệu


## Callbacks - Điều khiển quá trình huấn luyện

Callback là cách thêm logic vào quá trình train mà không cần tự viết lại `fit()`.

Ví dụ phổ biến:

- **EarlyStopping:** dừng train khi validation không cải thiện.
- **ModelCheckpoint:** lưu weights tốt nhất.
- **TensorBoard:** ghi log để trực quan hóa.
- **Custom callback:** tự định nghĩa hành vi riêng.

In [26]:
class EpochSummaryCallback(keras.callbacks.Callback):
    """Callback in tóm tắt loss sau mỗi epoch.

    Tham số:
        Không có.

    Kiểu trả về:
        None. Callback tạo tác dụng phụ bằng cách in log ra màn hình.
    """

    def on_epoch_end(self, epoch, logs=None):
        """Được Keras tự gọi khi một epoch kết thúc."""
        logs = logs or {}
        print(
            f"Epoch {epoch + 1}: "
            f"loss={logs.get('loss'):.4f}, "
            f"val_loss={logs.get('val_loss'):.4f}"
        )


callback_model = make_sequential_classifier()
callback_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

checkpoint_path = "C7_demo_best.weights.h5"
log_dir = "C7_demo_logs"

callbacks = [
    # Dừng sớm nếu validation loss không cải thiện sau 2 epoch.
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
    ),
    # Lưu bộ trọng số tốt nhất theo validation loss.
    keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
    ),
    # Ghi log để có thể xem bằng TensorBoard.
    keras.callbacks.TensorBoard(log_dir=log_dir),
    EpochSummaryCallback(),
]

history_cb = callback_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=callbacks,
    verbose=0,
)

print("Số epoch thực sự đã train:", len(history_cb.history["loss"]))
print("Weights tốt nhất được lưu tại:", checkpoint_path)
print("TensorBoard logs nằm trong thư mục:", log_dir)

Epoch 1: loss=1.0044, val_loss=0.9347
Epoch 2: loss=0.7868, val_loss=0.7894
Epoch 3: loss=0.6112, val_loss=0.6539
Epoch 4: loss=0.4609, val_loss=0.5348
Epoch 5: loss=0.3450, val_loss=0.4411
Epoch 6: loss=0.2629, val_loss=0.3754
Epoch 7: loss=0.2064, val_loss=0.3315
Epoch 8: loss=0.1672, val_loss=0.3016
Epoch 9: loss=0.1388, val_loss=0.2813
Epoch 10: loss=0.1173, val_loss=0.2667
Số epoch thực sự đã train: 10
Weights tốt nhất được lưu tại: C7_demo_best.weights.h5
TensorBoard logs nằm trong thư mục: C7_demo_logs


# 7.4 Tự viết vòng lặp huấn luyện và đánh giá

Khi `fit()` không đủ linh hoạt, ta có thể tự viết vòng lặp. Khi đó cần tự quản lý:

- Forward pass.
- Loss.
- Gradient.
- Cập nhật weights bằng optimizer.
- Metrics.
- Chế độ `training=True` hoặc `training=False`.

Đây là mức linh hoạt cao, nhưng cũng dễ lỗi hơn.

## Low-level usage of metrics - Dùng metric ở mức thấp

Khi dùng `fit()`, Keras tự cập nhật và reset metric. Khi tự viết loop, ta phải tự gọi `update_state()`, `result()` và `reset_state()`.

In [27]:
# Tạo metric object và tự cập nhật qua từng batch validation.

# Tạo metric SparseCategoricalAccuracy để tự tính accuracy.
# Dùng khi nhãn y là số nguyên, ví dụ 0, 1, 2 thay vì one-hot vector.
manual_acc = keras.metrics.SparseCategoricalAccuracy()

# Tạo validation dataset từ dữ liệu x_val và y_val.
# Sau đó chia dữ liệu thành các batch, mỗi batch có tối đa 64 mẫu.
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(64)

# Duyệt qua từng batch trong validation dataset.
for x_batch, y_batch in val_dataset:
    # training=False để model chạy ở chế độ suy luận.
    # Khi đó các layer như Dropout hoặc BatchNormalization sẽ hoạt động theo chế độ inference.
    y_pred = callback_model(x_batch, training=False)

    # Cập nhật metric accuracy bằng nhãn thật y_batch và dự đoán y_pred.
    manual_acc.update_state(y_batch, y_pred)

# Lấy kết quả accuracy cuối cùng sau khi đã duyệt qua toàn bộ validation dataset.
# manual_acc.result() trả về tensor, float(...) chuyển sang số Python để in dễ đọc.
print("Accuracy tính thủ công:", float(manual_acc.result()))

# Reset trạng thái của metric để có thể dùng lại cho lần tính tiếp theo.
manual_acc.reset_state()

Accuracy tính thủ công: 0.875


## Custom training loop với GradientTape

Một training step thủ công thường gồm:

1. Lấy batch dữ liệu.
2. Chạy forward pass trong `tf.GradientTape()`.
3. Tính loss.
4. Tính gradient theo trainable weights.
5. Optimizer cập nhật weights.
6. Cập nhật metric.

In [28]:
# Tạo một model phân loại dạng Sequential bằng hàm đã định nghĩa trước đó.
loop_model = make_sequential_classifier()

# Tạo optimizer Adam với learning rate là 0.001.
# Optimizer dùng để cập nhật trọng số của model trong quá trình train.
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

# Tạo hàm loss SparseCategoricalCrossentropy.
# Dùng khi nhãn y là số nguyên, ví dụ 0, 1, 2 thay vì one-hot vector.
loss_fn = keras.losses.SparseCategoricalCrossentropy()

# Tạo metric accuracy để theo dõi độ chính xác trong quá trình train.
train_acc_metric = keras.metrics.SparseCategoricalAccuracy()

# tf.data.Dataset giúp tạo pipeline batch dữ liệu cho vòng lặp thủ công.
# from_tensor_slices ghép từng mẫu x_train với nhãn y_train tương ứng.
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))

    # Xáo trộn dữ liệu train để model không học theo thứ tự cố định.
    .shuffle(buffer_size=800)

    # Chia dữ liệu thành các batch, mỗi batch có tối đa 64 mẫu.
    .batch(64)
)

# Train model thủ công trong 3 epoch.
for epoch in range(3):
    # Reset metric để accuracy của mỗi epoch được tính độc lập.
    train_acc_metric.reset_state()

    # Duyệt qua từng batch dữ liệu train.
    for x_batch, y_batch in train_dataset:
        # GradientTape ghi lại các phép tính để TensorFlow tự động lấy đạo hàm.
        with tf.GradientTape() as tape:
            # Gọi model ở chế độ training.
            # training=True để các layer như Dropout/BatchNormalization chạy theo chế độ train.
            y_pred = loop_model(x_batch, training=True)

            # Tính loss giữa nhãn thật và dự đoán của model.
            loss_value = loss_fn(y_batch, y_pred)

        # Tính gradient của loss theo các weight có thể huấn luyện.
        gradients = tape.gradient(loss_value, loop_model.trainable_weights)

        # Cập nhật weight bằng optimizer.
        # zip ghép từng gradient với weight tương ứng.
        optimizer.apply_gradients(zip(gradients, loop_model.trainable_weights))

        # Cập nhật metric bằng nhãn thật và xác suất dự đoán.
        train_acc_metric.update_state(y_batch, y_pred)

    # In loss của batch cuối cùng trong epoch và accuracy trung bình của cả epoch.
    print(
        f"Epoch {epoch + 1}: "
        f"loss={float(loss_value):.4f}, "
        f"accuracy={float(train_acc_metric.result()):.4f}"
    )

Epoch 1: loss=1.0635, accuracy=0.4787
Epoch 2: loss=0.7156, accuracy=0.6687
Epoch 3: loss=0.7469, accuracy=0.7738


## Evaluation loop và `tf.function`

Evaluation loop không cần `GradientTape` vì ta không cập nhật weights. Khi model có Dropout hoặc BatchNormalization, cần gọi model với `training=False`.

`tf.function` có thể biên dịch một step thành graph để chạy nhanh hơn. Nên debug bằng eager mode trước, sau đó mới thêm `tf.function`.

In [29]:
# Tạo metric accuracy để đánh giá model trên tập validation.
# Dùng SparseCategoricalAccuracy vì nhãn y là số nguyên, ví dụ 0, 1, 2.
eval_acc_metric = keras.metrics.SparseCategoricalAccuracy()


# @tf.function biên dịch hàm test_step thành graph TensorFlow.
# Điều này giúp chạy nhanh hơn so với Python function thông thường.
@tf.function
def test_step(x_batch, y_batch):
    """Thực hiện một bước đánh giá đã được TensorFlow biên dịch.

    Tham số:
        x_batch (tf.Tensor): batch đặc trưng validation.
        y_batch (tf.Tensor): batch nhãn thật.

    Kiểu trả về:
        None. Hàm cập nhật trực tiếp eval_acc_metric.
    """

    # Gọi model ở chế độ suy luận.
    # training=False để các layer như Dropout hoặc BatchNormalization chạy theo chế độ inference.
    y_pred = loop_model(x_batch, training=False)

    # Cập nhật metric accuracy bằng nhãn thật và dự đoán của model.
    eval_acc_metric.update_state(y_batch, y_pred)


# Chạy evaluation loop thủ công trên toàn bộ validation dataset.

# Reset metric trước khi đánh giá để không bị cộng dồn kết quả từ lần chạy trước.
eval_acc_metric.reset_state()

# Duyệt qua từng batch trong validation dataset.
for x_batch, y_batch in val_dataset:
    # Thực hiện một bước đánh giá cho batch hiện tại.
    test_step(x_batch, y_batch)

# In accuracy cuối cùng sau khi đã đánh giá toàn bộ tập validation.
print("Validation accuracy từ evaluation loop:", float(eval_acc_metric.result()))

Validation accuracy từ evaluation loop: 0.7723214030265808


## Tận dụng `fit()` với custom `train_step()`

Đây là lựa chọn trung gian rất hay:

- Ta tự viết logic cho **một training step**.
- Nhưng vẫn dùng được `fit()`, callbacks, logs, validation và history.

Nó phù hợp khi muốn tùy biến quá trình train nhưng không muốn tự viết toàn bộ epoch loop.

In [30]:
# Kế thừa keras.Model để tự định nghĩa cách train cho mỗi batch.
class CustomTrainStepModel(keras.Model):
    """Model tùy biến một bước train nhưng vẫn dùng được fit."""

    def train_step(self, data):
        """Định nghĩa logic cho một batch huấn luyện."""

        # data là một batch do fit truyền vào, gồm input và label.
        x_batch, y_batch = data

        # GradientTape dùng để ghi lại phép tính trong forward pass,
        # từ đó TensorFlow có thể tính gradient tự động.
        with tf.GradientTape() as tape:
            # Gọi model ở chế độ training.
            y_pred = self(x_batch, training=True)

            # Tính loss bằng loss đã khai báo trong compile().
            # regularization_losses=self.losses giúp cộng thêm regularization loss nếu có.
            loss = self.compiled_loss(
                y_batch,
                y_pred,
                regularization_losses=self.losses,
            )

        # Tính gradient của loss theo các trọng số trainable của model.
        gradients = tape.gradient(loss, self.trainable_weights)

        # Dùng optimizer đã khai báo trong compile() để cập nhật trọng số.
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))

        # Cập nhật các metric đã khai báo trong compile().
        self.compiled_metrics.update_state(y_batch, y_pred)

        # Trả về dict metric để Keras ghi log trong quá trình fit().
        return {metric.name: metric.result() for metric in self.metrics}


# Khai báo input của model, mỗi mẫu có num_features đặc trưng.
inputs = keras.Input(shape=(num_features,), name="features")

# Lớp Dense ẩn với 64 neuron và activation ReLU.
x_custom = layers.Dense(64, activation="relu")(inputs)

# Lớp output có num_classes neuron.
# softmax biến output thành xác suất cho từng class.
outputs = layers.Dense(num_classes, activation="softmax")(x_custom)

# Tạo model bằng class tùy biến CustomTrainStepModel.
custom_step_model = CustomTrainStepModel(
    inputs,
    outputs,
    name="custom_train_step_model"
)

# Compile model như bình thường.
# Optimizer, loss và metrics ở đây sẽ được dùng lại trong train_step().
custom_step_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

# Train model bằng fit().
# Dù đã custom train_step, ta vẫn dùng được fit như model Keras thông thường.
history_custom_step = custom_step_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=3,
    batch_size=32,
    verbose=0,
)

# In ra các key có trong history, ví dụ: loss, accuracy, val_loss, val_accuracy.
print("History keys:", history_custom_step.history.keys())

# In accuracy cuối cùng sau epoch cuối.
print("Final custom train_step accuracy:", history_custom_step.history["accuracy"][-1])

/home/zhu/micromamba/envs/tf/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:695: UserWarning: `model.compiled_loss()` is deprecated. Instead, use `model.compute_loss(x, y, y_pred, sample_weight, training)`.
  warnings.warn(
/home/zhu/micromamba/envs/tf/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:670: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_update_state(


History keys: dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])
Final custom train_step accuracy: 0.7475000023841858


# Tổng kết demo

Các điểm cần nhớ từ Chapter 7:

- **Sequential** phù hợp với model tuyến tính đơn giản.
- **Functional API** là lựa chọn trung tâm cho nhiều model thực tế: nhiều input, nhiều output, nhiều nhánh, shared layers.
- **Model subclassing** hữu ích khi kiến trúc cần logic Python tùy biến.
- **Built-in loops** (`compile`, `fit`, `evaluate`, `predict`) nên là lựa chọn mặc định.
- **Callbacks và custom metrics** giúp mở rộng `fit()` mà không cần tự viết loop.
- **Custom training loop** cho quyền kiểm soát cao nhất nhưng phải tự quản lý nhiều chi tiết.
- **Custom train_step** là điểm cân bằng: tùy biến một batch train nhưng vẫn tận dụng hệ sinh thái `fit()`.

Bài tập gợi ý cho sinh viên:

1. Thay đổi số layer/unit trong `Sequential` và quan sát `model.summary()`.
2. Thêm một output mới cho `ticket_model` và cấu hình loss tương ứng.
3. Viết một callback dừng train khi `val_accuracy` vượt một ngưỡng cho trước.
4. Thêm validation loss vào custom evaluation loop.
5. Viết lại `CustomTrainStepModel` để log thêm một metric tự định nghĩa.